
The `CREATE TABLE` statement in Hive is much more feature-rich than in traditional SQL because Hive is designed for **data warehousing on HDFS/object storage**. You can define **column types, partitions, buckets, storage format, compression, SerDe, table properties, and external locations**.

Below is the complete syntax followed by explanations and real-world examples.

---

# Complete Hive CREATE TABLE Syntax

```sql
CREATE [EXTERNAL] TABLE [IF NOT EXISTS] database_name.table_name
(
    column1 data_type [COMMENT 'comment'],
    column2 data_type,
    column3 data_type,
    ...
)
COMMENT 'Table description'

PARTITIONED BY
(
    partition_column1 data_type,
    partition_column2 data_type
)

CLUSTERED BY
(
    bucket_column1,
    bucket_column2
)
SORTED BY
(
    sort_column ASC,
    sort_column DESC
)
INTO number_of_buckets BUCKETS

ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
COLLECTION ITEMS TERMINATED BY '|'
MAP KEYS TERMINATED BY ':'
LINES TERMINATED BY '\n'
NULL DEFINED AS '\\N'

STORED AS ORC

LOCATION '/warehouse/sales'

TBLPROPERTIES
(
    'property1'='value1',
    'property2'='value2'
);
```

---

# Explanation of Each Attribute

| Attribute        | Purpose                                                  |
| ---------------- | -------------------------------------------------------- |
| `EXTERNAL`       | Data remains even if the table is dropped                |
| `IF NOT EXISTS`  | Prevents error if the table already exists               |
| `COMMENT`        | Adds metadata/documentation                              |
| `PARTITIONED BY` | Improves query performance by pruning partitions         |
| `CLUSTERED BY`   | Bucketing for joins and sampling                         |
| `SORTED BY`      | Sorts data within each bucket                            |
| `ROW FORMAT`     | Defines how the data is delimited                        |
| `STORED AS`      | Specifies the file format (ORC, PARQUET, TEXTFILE, etc.) |
| `LOCATION`       | Custom HDFS/S3/ADLS path                                 |
| `TBLPROPERTIES`  | Additional metadata and settings                         |

---

# Example 1: Simple Managed Table

```sql
CREATE TABLE employee
(
    emp_id INT,
    emp_name STRING,
    department STRING,
    salary DOUBLE,
    joining_date DATE
)
STORED AS PARQUET;
```

---

# Example 2: External Table

```sql
CREATE EXTERNAL TABLE employee_ext
(
    emp_id INT,
    emp_name STRING,
    salary DOUBLE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','

STORED AS TEXTFILE

LOCATION '/data/employee';
```

**Why External?**

* Data is stored outside Hive's warehouse directory.
* Dropping the table removes only metadata.
* The files remain in storage.

---

# Example 3: Partitioned Table

```sql
CREATE TABLE sales
(
    order_id BIGINT,
    customer_id BIGINT,
    amount DECIMAL(10,2)
)
PARTITIONED BY
(
    year INT,
    month INT
)
STORED AS ORC;
```

Example partitions:

```
sales/

year=2025/
    month=1/

year=2025/
    month=2/

year=2026/
    month=1/
```

Queries like:

```sql
SELECT *
FROM sales
WHERE year=2026
AND month=7;
```

scan only the matching partition.

---

# Example 4: Bucketed Table

```sql
CREATE TABLE customer
(
    customer_id BIGINT,
    name STRING,
    city STRING
)
CLUSTERED BY (customer_id)
INTO 8 BUCKETS
STORED AS ORC;
```

Used for:

* Efficient joins
* Sampling
* Better distribution of data

---

# Example 5: Partition + Bucket

```sql
CREATE TABLE transactions
(
    txn_id BIGINT,
    customer_id BIGINT,
    amount DOUBLE
)
PARTITIONED BY
(
    txn_year INT
)
CLUSTERED BY
(
    customer_id
)
INTO 16 BUCKETS
STORED AS PARQUET;
```

---

# Example 6: Delimited Text File

```sql
CREATE TABLE student
(
    id INT,
    name STRING,
    marks INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;
```

Input file

```
1,John,90
2,Alice,95
3,Bob,80
```

---

# Example 7: ORC Table with Compression

```sql
CREATE TABLE orders
(
    order_id BIGINT,
    amount DOUBLE
)
STORED AS ORC
TBLPROPERTIES
(
    'orc.compress'='SNAPPY'
);
```

---

# Example 8: Table Comments

```sql
CREATE TABLE employee
(
    emp_id INT COMMENT 'Employee ID',
    emp_name STRING COMMENT 'Employee Name',
    salary DOUBLE COMMENT 'Monthly Salary'
)
COMMENT 'Employee Master Table';
```

---

# Example 9: Custom Location

```sql
CREATE EXTERNAL TABLE logs
(
    log_id BIGINT,
    message STRING
)
STORED AS PARQUET

LOCATION '/data/application_logs/';
```

---

# Common Storage Formats

| Format       | Description            | Best For                     |
| ------------ | ---------------------- | ---------------------------- |
| TEXTFILE     | Plain text             | Small datasets, raw data     |
| ORC          | Optimized Row Columnar | Hive workloads               |
| PARQUET      | Columnar format        | Spark, Hive, Impala          |
| AVRO         | Row-based with schema  | Streaming, Kafka integration |
| SEQUENCEFILE | Hadoop binary          | Legacy Hadoop jobs           |

---

# Common Data Types

| Type          | Example                      |
| ------------- | ---------------------------- |
| TINYINT       | 10                           |
| SMALLINT      | 100                          |
| INT           | 1000                         |
| BIGINT        | 123456789                    |
| FLOAT         | 12.5                         |
| DOUBLE        | 25.78                        |
| DECIMAL(10,2) | 12345.67                     |
| STRING        | 'John'                       |
| VARCHAR(50)   | 'Alice'                      |
| CHAR(10)      | 'HR'                         |
| BOOLEAN       | TRUE                         |
| DATE          | '2026-07-23'                 |
| TIMESTAMP     | '2026-07-23 10:30:00'        |
| ARRAY         | `ARRAY<STRING>`              |
| MAP           | `MAP<STRING,STRING>`         |
| STRUCT        | `STRUCT<id:INT,name:STRING>` |

---

# Managed vs External Table

| Feature                 | Managed                   | External                 |
| ----------------------- | ------------------------- | ------------------------ |
| Hive owns data          | ✅                         | ❌                        |
| Data location           | Hive warehouse            | Custom location          |
| Drop table deletes data | ✅                         | ❌                        |
| Used for                | Internal warehouse tables | Shared/external datasets |

---

# Interview-Ready Example

```sql
CREATE EXTERNAL TABLE sales_fact
(
    sale_id BIGINT,
    customer_id BIGINT,
    product_id BIGINT,
    quantity INT,
    amount DECIMAL(12,2),
    sale_date DATE
)
PARTITIONED BY
(
    year INT,
    month INT
)
CLUSTERED BY
(
    customer_id
)
INTO 8 BUCKETS
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS PARQUET
LOCATION '/warehouse/sales/'
TBLPROPERTIES
(
    'parquet.compression'='SNAPPY'
);
```

This example combines many of the features commonly discussed in interviews: an **external table**, **partitioning**, **bucketing**, **Parquet storage**, a **custom location**, and **compression settings**.

> **Tip:** In modern Hive and Spark data lakes, **Parquet** or **ORC** are generally preferred over `TEXTFILE` because they are columnar formats that support compression, predicate pushdown, and faster analytical queries. Partitioning is used much more frequently than bucketing in production environments, while bucketing is typically applied only when there is a specific need such as optimizing certain joins or sampling.


## STORED AS TEXTFILE and CSV

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_learning.my_raw_feed.dummy_table (
    ID INT,
    TODAY_DATE DATE,
    DAY STRING,
    TIMESTAMP_1 TIMESTAMP,
    COL1 STRING,
    COL2 STRING,
    COL3 STRING
)

PARTITIONED BY  (YEAR STRING, MONTH STRING,REPORTING_DATE STRING)
CLUSTERED BY (ID) 
SORTED BY  (TODAY ASC)
INTO 10 BUCKETS

ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'

STORED AS TEXTFILE
LOCATION '/Volumes/my_learning/my_raw_feed/my_volume/'




# Stored as Parquet,avro,orc

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS my_learning.my_raw_feed.dummy_table (
    ID INT,
    TODAY_DATE DATE,
    DAY STRING,
    TIMESTAMP_1 TIMESTAMP,
    COL1 STRING,
    COL2 STRING,
    COL3 STRING
)

PARTITIONED BY  (YEAR STRING, MONTH STRING,REPORTING_DATE STRING)
CLUSTERED BY (ID) 
SORTED BY  (TODAY ASC)
INTO 10 BUCKETS

STORED AS PARQUET
LOCATION 'dbfs:/Volumes/my_learning/my_raw_feed/my_volume/'

TBLPROPERTIES(
    parquet.compression.codec = 'snappy'
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_learning.my_raw_feed.dummy_table (
    ID INT,
    TODAY_DATE DATE,
    DAY STRING,
    MONTH STRING,
    TIMESTAMP_1 TIMESTAMP,
    COL1 STRING,
    COL2 STRING,
    COL3 STRING
)

PARTITIONED BY  (YEAR STRING, MONTH STRING,REPORTING_DATE STRING)
CLUSTERED BY (ID) 
SORTED BY  (TODAY ASC)
INTO 10 BUCKETS

ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\n'

STORED AS TEXTFILE
LOCATION '/Volumes/my_learning/my_raw_feed/my_volume/'





If you're creating a **Parquet table**, there's an important point to know:

> **`ROW FORMAT DELIMITED FIELDS TERMINATED BY` is used only for text-based files (TEXTFILE, CSV, TSV).**
> **Parquet is a binary columnar format**, so it **does not use delimiters**.

However, if your interviewer asks for a "complete" Hive `CREATE TABLE` syntax, you can show the `ROW FORMAT DELIMITED` clause for text files and explain that it is **not applicable to Parquet**.

---

## 1. TEXTFILE Table (Uses ROW FORMAT DELIMITED)

```sql
CREATE EXTERNAL TABLE employee
(
    emp_id          INT,
    emp_name        STRING,
    department      STRING,
    salary          DECIMAL(10,2),
    joining_date    DATE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
COLLECTION ITEMS TERMINATED BY '|'
MAP KEYS TERMINATED BY ':'
LINES TERMINATED BY '\n'
NULL DEFINED AS '\\N'
STORED AS TEXTFILE
LOCATION '/data/employee/';
```

This is used when your input file looks like:

```text
101,John,IT,50000,2024-01-01
102,Alice,HR,60000,2024-01-02
```

---

## 2. Parquet Table (Recommended)

```sql
CREATE EXTERNAL TABLE employee
(
    emp_id          INT,
    emp_name        STRING,
    department      STRING,
    salary          DECIMAL(10,2),
    joining_date    DATE
)
STORED AS PARQUET
LOCATION '/warehouse/employee/';
```

Notice there is **no**:

* `ROW FORMAT DELIMITED`
* `FIELDS TERMINATED BY`
* `LINES TERMINATED BY`

because Parquet stores data in a structured binary format.

---

## 3. Loading CSV into a Parquet Table (Typical ETL)

### Step 1: Create a staging table for CSV

```sql
CREATE EXTERNAL TABLE employee_stg
(
    emp_id          INT,
    emp_name        STRING,
    department      STRING,
    salary          DECIMAL(10,2),
    joining_date    DATE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/staging/employee/';
```

### Step 2: Create the Parquet table

```sql
CREATE TABLE employee_parquet
(
    emp_id          INT,
    emp_name        STRING,
    department      STRING,
    salary          DECIMAL(10,2),
    joining_date    DATE
)
STORED AS PARQUET;
```

### Step 3: Convert CSV to Parquet

```sql
INSERT INTO employee_parquet
SELECT *
FROM employee_stg;
```

Hive reads the delimited text from the staging table and writes it out as Parquet.

---

# Interview Question

**Q: Can we use `ROW FORMAT DELIMITED` with `STORED AS PARQUET`?**

**Answer:**

Generally, **no**. `ROW FORMAT DELIMITED` is intended for delimited text formats such as `TEXTFILE`. Parquet files are self-describing binary files, so delimiters like `FIELDS TERMINATED BY ','` are not used.

---

# Quick Comparison

| Storage Format                 | `ROW FORMAT DELIMITED` | `FIELDS TERMINATED BY` |
| ------------------------------ | ---------------------- | ---------------------- |
| TEXTFILE                       | ✅ Yes                  | ✅ Yes                  |
| CSV (via TEXTFILE + delimiter) | ✅ Yes                  | ✅ Yes                  |
| ORC                            | ❌ No                   | ❌ No                   |
| PARQUET                        | ❌ No                   | ❌ No                   |
| AVRO                           | ❌ No                   | ❌ No                   |

**Production best practice:** In most data lake architectures, raw CSV data is first ingested into a **TEXTFILE staging table** using `ROW FORMAT DELIMITED`, then transformed and stored in **Parquet** (or ORC) tables for efficient querying.
